# Preparación datos para DashBoard

## Import y rutas

In [11]:
from pathlib import Path
import pandas as pd

import numpy as np


In [12]:
ROOT_DIR = Path.cwd().parent
RAW_DATA_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

RAW_KUEPA_FILE = RAW_DATA_DIR / "Prueba_Tecnica_Analista_Datos_EFE.xlsx" 
PROCESSED_DATA_INCRIPCION = PROCESSED_DATA_DIR / "Inscritos.csv"
PROCESSED_DATA_ACTIVIDAD = PROCESSED_DATA_DIR / "Actividad.csv"
PROCESSED_DATA_EMPLEABILIDAD = PROCESSED_DATA_DIR / "Empleabilidad.csv"

DF_DASHBOARD = PROCESSED_DATA_DIR / "df_dashboard.parquet"
DF_TASAS = PROCESSED_DATA_DIR / "df_tasas_corregidas.parquet"
DF_EMPLEABILIDAD = PROCESSED_DATA_DIR / "df_empleabilidad_resumen.parquet"


## Carga

In [13]:
df_inscritos=pd.read_csv(PROCESSED_DATA_INCRIPCION)
print(f"Dataset loaded with {df_inscritos.shape[0]} rows and {df_inscritos} columns")

df_actividad = pd.read_csv(PROCESSED_DATA_ACTIVIDAD)
print(f"Dataset loaded with {df_inscritos.shape[0]} rows and {df_inscritos.shape[1]} columns")

df_empleabilidad = pd.read_csv(PROCESSED_DATA_EMPLEABILIDAD)
print(f"Dataset loaded with {df_empleabilidad.shape[0]} rows and {df_empleabilidad.shape[1]} columns")


Dataset loaded with 722 rows and     id_inscrito   documento                     nombre genero  edad  \
0      KP-00670  1081624071        Camila Ortiz Pineda      F  20.0   
1      KP-00557  1060304725        Angie Ramírez Gómez      F  25.0   
2      KP-00462  1086701248      Cristian Ortiz Torres      F  27.0   
3      KP-00590  1048716090       Steven López Ramírez      F  30.0   
4      KP-00165  1099940210           Juan Muñoz Gómez      F  18.0   
..          ...         ...                        ...    ...   ...   
717    KP-00561  1036069609         Felipe Torres Díaz      F  30.0   
718    KP-00125  1039613958    Laura Quintero Quintero      M  24.0   
719    KP-00643  1084447247        Jhon Rojas Quintero      F  27.0   
720    KP-00484  1090058234        Steven Moreno López      F  23.0   
721    KP-00124  1084632920  Camila Hernández Quintero      F  21.0   

           ciudad           proyecto canal_convocatoria gestor  estrato  \
0          Bogotá      Impulsa Mujer   

In [14]:
df_actividad['fecha_primer_ingreso'] = pd.to_datetime(
    df_actividad['fecha_primer_ingreso'],
    errors='coerce'
)
df_actividad['ultima_conexion'] = pd.to_datetime(
    df_actividad['ultima_conexion'],
    errors='coerce'
)
df_inscritos['fecha_inscripcion'] = pd.to_datetime(
    df_inscritos['fecha_inscripcion'],
    errors='coerce'
)

df_empleabilidad['fecha_colocacion'] = pd.to_datetime(
    df_empleabilidad['fecha_colocacion'],
    errors='coerce'
)

In [15]:

df_actividad.drop(labels='dias_sin_conexion', axis=1,  inplace=True, errors='raise')


## Estadisticos

In [16]:
#Definir fecha de corte
FECHA_CORTE = pd.to_datetime('2026-09-30')

#  Preparar df_actividad (Cálculos de riesgo y avance)
df_actividad['dias_sin_conexion'] = (FECHA_CORTE - df_actividad['ultima_conexion']).dt.days

# Unir inscritos con actividad para crear el DataFrame maestro del dashboard
df_dashboard = df_inscritos.merge(df_actividad, on='id_inscrito', how='left')

#  Limpieza de nulos en columnas numéricas tras el merge
df_dashboard['porcentaje_avance'] = df_dashboard['porcentaje_avance'].fillna(0)
df_dashboard['modulos_completados'] = df_dashboard['modulos_completados'].fillna(0)

# Si no hay fecha de última conexión, usamos la fecha de inscripción como referencia inicial
df_dashboard['dias_sin_conexion'] = df_dashboard['dias_sin_conexion'].fillna(
    (FECHA_CORTE - pd.to_datetime(df_dashboard['fecha_inscripcion'])).dt.days
)

#  Lógica de "Graduado Validado" 
# Validamos los 3 campos: Estado, Módulos (8) y Porcentaje (>=99%)
df_dashboard['es_graduado_validado'] = (
    (df_dashboard['estado'] == 'Graduado') &
    (df_dashboard['modulos_completados'] == 8) &
    (df_dashboard['porcentaje_avance'] >= 99)
).astype(int)

# Calcular tasas de graduación corregidas por Proyecto y Canal
df_tasas_corregidas = df_dashboard.groupby(['proyecto', 'canal_convocatoria']).agg(
    total_inscritos=('id_inscrito', 'count'),
    graduados_validados=('es_graduado_validado', 'sum')
).reset_index()

df_tasas_corregidas['tasa_graduacion_real'] = (
    df_tasas_corregidas['graduados_validados'] / df_tasas_corregidas['total_inscritos'] * 100
).round(1)
# Resumen de empleabilidad 
df_empleabilidad_resumen = df_empleabilidad.groupby('id_inscrito').agg(
    colocado=('colocado', 'first'),
    salario_promedio=('salario', 'mean')
).reset_index()

In [17]:
# Guardar los DataFrames procesados
df_dashboard.to_parquet(DF_DASHBOARD, index=False)
df_tasas_corregidas.to_parquet(DF_TASAS, index=False)
df_empleabilidad_resumen.to_parquet(DF_EMPLEABILIDAD, index=False)

print("Datos guardados exitosamente.")

Datos guardados exitosamente.
